In [1]:
import pandas as pd

df = pd.read_parquet("data/bpi2017.parquet")
case, act, ts = "case:concept:name", "concept:name", "time:timestamp"

# Application steps, finished apps only
a = df[df["EventOrigin"] == "Application"].sort_values([case, ts]).copy()
ends = ["A_Pending", "A_Cancelled", "A_Denied"]
closed = a.loc[a[act].isin(ends), case].unique()
a = a[a[case].isin(closed)].copy()
print("Finished apps used:", len(closed))

# Days from each step to the next one
a["next_step"] = a.groupby(case)[act].shift(-1)
a["wait_days"] = (a.groupby(case)[ts].shift(-1) - a[ts]).dt.total_seconds() / 86400
w = a.dropna(subset=["next_step"])

summary = (
    w.groupby([act, "next_step"])["wait_days"]
    .agg(count="count", median="median",
         p90=lambda s: s.quantile(0.9), total="sum")
    .query("count >= 500")
    .sort_values("total", ascending=False)
)
summary["share_%"] = summary["total"] / w["wait_days"].sum() * 100

print("\nTop 10 waits, ranked by total days lost:")
print(summary.round(1).head(10).to_string())

Finished apps used: 31411

Top 10 waits, ranked by total days lost:
                           count  median   p90     total  share_%
concept:name next_step                                           
A_Complete   A_Cancelled    9307    30.7  30.9  263957.9     38.5
             A_Validating  21825     7.8  18.0  214193.3     31.2
A_Incomplete A_Validating  16894     1.0   7.0   48192.8      7.0
A_Concept    A_Accepted    31411     0.9   3.1   44272.2      6.5
A_Validating A_Incomplete  22965     0.9   5.0   38305.1      5.6
A_Incomplete A_Pending      4937     3.7  15.7   29023.8      4.2
A_Validating A_Pending     12291     0.7   4.2   17679.3      2.6
A_Incomplete A_Cancelled     924     7.4  35.9   14003.3      2.0
A_Validating A_Denied       3373     2.0   6.0    9451.3      1.4
A_Accepted   A_Complete    31264     0.0   0.0    4098.4      0.6


In [2]:
c = w[(w[act] == "A_Complete") & (w["next_step"] == "A_Cancelled")]["wait_days"]

print(c.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2))
print("\nShare cancelled between 29 and 32 days:",
      round(c.between(29, 32).mean() * 100, 1), "%")
print("Share cancelled before 29 days:",
      round((c < 29).mean() * 100, 1), "%")

count    9307.00
mean       28.36
std        10.05
min         0.00
1%          0.04
5%          4.06
25%        30.54
50%        30.70
75%        30.85
95%        38.47
99%        55.97
max        79.70
Name: wait_days, dtype: float64

Share cancelled between 29 and 32 days: 77.8 %
Share cancelled before 29 days: 14.6 %


In [3]:
start = a.groupby(case)[ts].min()
end_rows = a[a[act].isin(ends)]
end_time = end_rows.groupby(case)[ts].min()
outcome = end_rows.groupby(case)[act].first()
online = a.groupby(case)[act].apply(lambda s: "A_Submitted" in set(s))

e2e = pd.DataFrame({
    "days": (end_time - start).dt.total_seconds() / 86400,
    "outcome": outcome,
    "online": online,
    "start": start,
})

cutoff = pd.Timestamp("2016-10-01", tz="UTC")
early = e2e[e2e["start"] < cutoff]

print("All finished apps:", len(e2e), "| median days:", round(e2e["days"].median(), 1))
print("Started before Oct 2016:", len(early), "| median days:", round(early["days"].median(), 1))

print("\nBy outcome (before Oct 2016):")
print(early.groupby("outcome")["days"].agg(["count", "median"]).round(1))

print("\nBy channel (True = online, before Oct 2016):")
print(early.groupby("online")["days"].agg(["count", "median"]).round(1))

All finished apps: 31411 | median days: 19.1
Started before Oct 2016: 23472 | median days: 19.5

By outcome (before Oct 2016):
             count  median
outcome                   
A_Cancelled   7871    31.7
A_Denied      2703    14.6
A_Pending    12898    14.9

By channel (True = online, before Oct 2016):
        count  median
online               
False    8281    15.9
True    15191    21.9


In [4]:
approved = early[early["outcome"] == "A_Pending"]
print("Approved apps only, by channel (True = online):")
print(approved.groupby("online")["days"].agg(["count", "median"]).round(1))

Approved apps only, by channel (True = online):
        count  median
online               
False    5376    13.8
True     7522    16.0


## Where applications get stuck

I measured the days between each step and the next one, using the 31,411 applications that reached a final outcome. Then I ranked each step-to-step move by the total days it added up across all applications.

The biggest delay isn't inside the bank at all. Most of the waiting happens after the bank has sent an offer and is waiting on the customer. Two moves account for about 70% of all waiting time: an offer that ends in cancellation, and an offer that eventually comes back for validation (typically 8 days, sometimes 18 or more).

The cancellations stood out. 78% of them happen between 29 and 32 days after the offer, and the middle half fall within about seven hours of each other. People don't behave that consistently, so this looks like an automatic timeout around 30 days. I can't confirm the bank's actual rule from the data, but the pattern is very clear. Around 7,200 applications sat idle for a month before closing this way.

The bank's own steps are quick. Checking an application, accepting it and validating documents usually takes less than a day.

A typical application takes about 19.5 days from start to finish. Approved and denied applications both take around 15 days, while cancelled ones take about 32, which is the timeout again.

At first online applications looked 6 days slower than staff-entered ones. Most of that gap comes from online applications being cancelled more often. Comparing approved applications only, online ones are about 2 days slower.

I checked whether applications from late 2016 skew these numbers, since they had less time to finish. Using only applications started before October barely changed the result (19.5 days vs 19.1).

If I were advising the bank, I'd start with the waiting period after an offer is sent. A reminder in the first two weeks would likely do more than speeding up internal checks.